# 🧠 JeBert: Decoupled Dual-Model Hybrid Named Entity Recognition

**JeBert** couples high-recall neural candidate extraction (**SpanNER**) with independent semantic reasoning (**TypeSafe System One Jev**), smart uncertainty gating, and escalation arbitration.

### 🔄 Decoupled Step-by-Step Architecture Pipeline:
1. **Stage 1 (SpanNER)**: Neural dense span proposal and candidate hypothesis generation (cached).
2. **Stage 2 (Jev System One)**: Pure independent semantic classification on candidate boundary spans (without SpanNER labels).
3. **Stage 3 (Dual-Model Gating)**: Discrepancy, uncertainty, and novelty gating separating consensus spans (0 cost) from escalated spans.
4. **Stage 4 (Escalation Arbitration)**: Resolves difficult/conflicting spans via LLM arbitration or calibrated fast dual-model referee + Margin Safeguard Veto.
5. **Stage 5 (Dual-View Benchmark)**: Candidate-level decision accuracy and global end-to-end dataset evaluation.


In [1]:
# ==============================================================================
# [CELL 1]: Setup, Environment & Imports
# ==============================================================================
import os
import time
import json
import asyncio
import httpx
import shutil
import concurrent.futures
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter

from src.common.config import PROJECT_ROOT, OUTPUT_DIR, JEBERT_DIR
from src.common.label_mapping import canonicalize_label
from src.data.cached_loader import load_cached_spanner_candidates
from src.common.evaluation_framework import evaluate_ner_dual_view
from src.common.audit_logger import (
    compute_canonical_audit_breakdown,
    format_canonical_audit_table,
    log_jebert_abstract_audit,
)
from src.architectures.JeBert import (
    JeBertPipeline,
    JeBertGating,
    JeBertJevEvaluator,
    JeBertArbitrator,
)

# Structured Output Directories under output/jebert/ (mirrors output/bertized_ace/)
JEBERT_RUNS_DIR = JEBERT_DIR / "runs"
JEBERT_CACHE_DIR = JEBERT_DIR / "cache"
JEBERT_RUNS_DIR.mkdir(parents=True, exist_ok=True)
JEBERT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("✅ JeBert environment & dependencies loaded successfully.")
print(f"📁 JeBert Structured Output Directory: {JEBERT_DIR}")



✅ JeBert environment & dependencies loaded successfully.


In [2]:
# ==============================================================================
# [CELL 2]: Load SpanNER Candidate Spans & Ground Truth (SpanNER View)
# ==============================================================================
CACHED_SPANNER_DIR = PROJECT_ROOT / 'data' / 'cached' / 'spanner_160_10_class'
VAL_DATA_DIR = PROJECT_ROOT / 'data' / 'new' / '400Abstracts' / 'no_disease' / 'val'
if not VAL_DATA_DIR.exists():
    VAL_DATA_DIR = PROJECT_ROOT / 'data' / 'new' / '400Abstracts' / 'val'

print('=' * 80)
print('📦 Loading 400/val Candidate Spans from Cache...')
print(f'  • Candidate Cache Source: {CACHED_SPANNER_DIR / "400" / "val"}')
print(f'  • Ground Truth Directory: {VAL_DATA_DIR}')
print('=' * 80)

# 1. Load candidate spans using cached loader
test_candidates_cache = load_cached_spanner_candidates(
    split_key='400/val',
    cache_dir=str(CACHED_SPANNER_DIR),
)

# 2. Attach ground truth entities from .ann files
def load_ann_ground_truth(ann_path):
    gt_ents = []
    if not os.path.exists(ann_path):
        return gt_ents
    with open(ann_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or not line.startswith('T'):
                continue
            parts = line.split('\t')
            if len(parts) < 3:
                continue
            type_offsets = parts[1]
            ent_text = parts[2]
            type_parts = type_offsets.split()
            lbl = type_parts[0]
            raw_offsets = type_offsets[len(lbl):].strip().replace(';', ' ').split()
            if len(raw_offsets) >= 2:
                gt_ents.append({
                    'label': canonicalize_label(lbl, use_new_labels=True),
                    'start_char': int(raw_offsets[0]),
                    'end_char': int(raw_offsets[-1]),
                    'text': ent_text
                })
    return gt_ents

for doc_id, doc_data in test_candidates_cache.items():
    ann_file = VAL_DATA_DIR / f'{doc_id}.ann'
    doc_data['ground_truth'] = load_ann_ground_truth(ann_file)
    doc_data['gt_ents'] = doc_data['ground_truth']

# 3. Build SpanNER predictions view and UNLABELED candidate boundary mentions for Jev
spanner_preds_by_doc = {}
unlabeled_candidates_by_doc = {}

for doc_id, doc_data in test_candidates_cache.items():
    raw_cands = doc_data.get('candidates', [])
    spanner_list = []
    unlabeled_list = []
    for idx, c in enumerate(raw_cands):
        cid = c.get('span_id', f'{doc_id}_s{idx}')
        s_text = c.get('span_text', c.get('entity', '')).strip()
        s_lbl = canonicalize_label(
            c.get('bert_predicted_label', c.get('predicted_label', 'O')),
            use_new_labels=True
        )
        # SpanNER feature record
        spanner_list.append({
            'id': cid,
            'span_text': s_text,
            'start_char': c.get('start_char', c.get('start', 0)),
            'end_char': c.get('end_char', c.get('end', 0)),
            'spanner_label': s_lbl,
            'bert_predicted_label': s_lbl,
            'label': s_lbl,
            'margin': float(c.get('margin', 0.0)),
            'uncertainty': float(c.get('uncertainty', c.get('u_score', 0.0))),
            'p_o': float(c.get('p_o', c.get('p_background_o', 0.0))),
            'novelty_score': float(c.get('novelty_score', c.get('novelty', 0.0))),
        })
        # Pure boundary span for Jev (Zero SpanNER labels passed)
        unlabeled_list.append({
            'id': cid,
            'span_text': s_text,
            'start_char': c.get('start_char', c.get('start', 0)),
            'end_char': c.get('end_char', c.get('end', 0)),
        })
    spanner_preds_by_doc[doc_id] = spanner_list
    unlabeled_candidates_by_doc[doc_id] = unlabeled_list

total_cands = sum(len(v) for v in spanner_preds_by_doc.values())
total_gts = sum(len(v['ground_truth']) for v in test_candidates_cache.values())
spanner_class_counts = Counter(c['spanner_label'] for c_list in spanner_preds_by_doc.values() for c in c_list)

print(f'✅ Successfully loaded {len(test_candidates_cache)} abstracts:')
print(f'  • Total candidate spans proposed: {total_cands} ({total_cands/max(1, len(test_candidates_cache)):.1f} spans/abstract)')
print(f'  • Total Ground Truth entities   : {total_gts} ({total_gts/max(1, len(test_candidates_cache)):.1f} entities/abstract)')
print('\n📊 SpanNER Candidate Proposal Distribution:')
for cls, cnt in spanner_class_counts.most_common():
    print(f'  • {cls:<48}: {cnt:4d} ({cnt/total_cands*100:5.1f}%)')


📦 Loading 400/val Candidate Spans from Cache...
  • Candidate Cache Source: /Users/hmd/Documents/Workspace/Unified_Test/data/cached/spanner_160_10_class/400/val
  • Ground Truth Directory: /Users/hmd/Documents/Workspace/Unified_Test/data/new/400Abstracts/no_disease/val
✅ Successfully loaded 80 abstracts:
  • Total candidate spans proposed: 1492 (18.6 spans/abstract)
  • Total Ground Truth entities   : 1422 (17.8 entities/abstract)

📊 SpanNER Candidate Proposal Distribution:
  • Nutrition                                       :  509 ( 34.1%)
  • Sleep                                           :  184 ( 12.3%)
  • Substance_use                                   :  167 ( 11.2%)
  • Environmental_exposures                         :  164 ( 11.0%)
  • Socioeconomic_factors                           :  158 ( 10.6%)
  • Physical_activities                             :  135 (  9.0%)
  • Non_physical_leisure_time_activities            :   71 (  4.8%)
  • Personal_care_products_and_cosmetic_proce

In [3]:
# ==============================================================================
# [CELL 3]: Independent TypeSafe Jev Semantic Classification (Jev View)
# ==============================================================================
# Jev evaluates candidate boundary spans purely on semantic criteria and domain guidelines,
# without receiving any labels or hypotheses from SpanNER.
JEV_MODE = "two_stage"      # "two_stage" (Dynamic rule routing), "mode2" (helpful), "mode1" (direct LSF)
GUIDEBOOK_PATH = str(PROJECT_ROOT / "output" / "bertized_ace" / "runs" / "run_20260925_110445_bertized_ace_16k" / "dynamicGuidebook.json")
CONCURRENCY = 8
USE_CACHE_IF_AVAILABLE = True  # Instant offline loading if cache exists; set False for live inference

jev_evaluator = JeBertJevEvaluator(
    mode=JEV_MODE,
    unbiased=True,           # Candidate spans evaluated without SpanNER labels
    guidebook_path=GUIDEBOOK_PATH,
    max_assigned_rules=4,
)

print("=" * 80)
print(f"⚡ Running Async Jev Semantic Classification ({JEV_MODE}) on {len(test_candidates_cache)} abstracts...")
print(f"   Guidebook: {Path(GUIDEBOOK_PATH).name} ({len(jev_evaluator.active_rules)} rules loaded)")
print("=" * 80)

t0_jev = time.time()
jev_preds_by_doc = {}

# Check for pre-computed cache across local directories
cache_candidates = [
    JEBERT_CACHE_DIR / "bertized_ace_checkpoint_cache.json",
    PROJECT_ROOT / "output" / "bertized_ace" / "cache" / "bertized_ace_checkpoint_cache.json",
    PROJECT_ROOT / "Jev" / "results" / "bertized_ace_checkpoint_cache.json",
]
cache_file = next((p for p in cache_candidates if p.exists()), None)
loaded_from_cache = False

if USE_CACHE_IF_AVAILABLE and cache_file:
    try:
        with open(cache_file, "r", encoding="utf-8") as f:
            cache_data = json.load(f)
        for doc_id in test_candidates_cache:
            if doc_id in cache_data:
                jev_preds_by_doc[doc_id] = cache_data[doc_id]
        if len(jev_preds_by_doc) == len(test_candidates_cache):
            loaded_from_cache = True
            print(f"📦 Loaded pre-computed Jev checkpoint cache ({len(jev_preds_by_doc)} abstracts loaded from {cache_file.name}).")
    except Exception as e:
        print(f"⚠️ Cache read warning: {e}. Falling back to live inference.")

if not loaded_from_cache:
    async def _run_jev_batch():
        sem = asyncio.Semaphore(CONCURRENCY)
        total_tasks = len(test_candidates_cache)
        completed = 0
        
        async with httpx.AsyncClient(timeout=45.0) as http_client:
            async def _eval_doc(doc_id, doc_info):
                nonlocal completed
                abstract_text = doc_info.get("abstract", doc_info.get("text", ""))
                # Pure candidate spans WITHOUT SpanNER labels
                cands = unlabeled_candidates_by_doc[doc_id]
                async with sem:
                    doc_preds = await jev_evaluator.evaluate_spans_async(
                        abstract=abstract_text,
                        candidate_spans=cands,
                        http_client=http_client,
                    )
                completed += 1
                if completed % 10 == 0 or completed == total_tasks:
                    print(f"   [{completed:2d}/{total_tasks:2d}] abstracts evaluated ({completed/total_tasks*100:5.1f}%)...")
                return doc_id, doc_preds

            tasks = [_eval_doc(did, dinfo) for did, dinfo in test_candidates_cache.items()]
            results = await asyncio.gather(*tasks)
            for did, preds in results:
                jev_preds_by_doc[did] = preds

    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor() as pool:
            pool.submit(lambda: asyncio.run(_run_jev_batch())).result()
    else:
        asyncio.run(_run_jev_batch())

    # Cache results into output/jebert/cache/ for instant reuse
    try:
        with open(JEBERT_CACHE_DIR / "bertized_ace_checkpoint_cache.json", "w", encoding="utf-8") as f:
            json.dump(jev_preds_by_doc, f, indent=2)
        print(f"💾 Cached Jev evaluations to: {JEBERT_CACHE_DIR / 'bertized_ace_checkpoint_cache.json'}")
    except Exception as e:
        pass

elapsed_jev = time.time() - t0_jev
n_docs = max(1, len(jev_preds_by_doc))
total_jev_spans = sum(len(v) for v in jev_preds_by_doc.values())
jev_class_counts = Counter(p.get("label", "O") for p_list in jev_preds_by_doc.values() for p in p_list)

print(f"\n✅ Jev Semantic Classification complete in {elapsed_jev:.2f}s ({elapsed_jev/n_docs:.2f}s/doc, {elapsed_jev/max(1, total_jev_spans)*1000:.1f}ms/span)!")
print("\n📊 Jev Independent Prediction Distribution:")
for cls, cnt in jev_class_counts.most_common():
    print(f"  • {cls:<48}: {cnt:4d} ({cnt/total_jev_spans*100:5.1f}%)")



⚡ Running Async Jev Semantic Classification (two_stage) on 80 abstracts...
   Guidebook: dynamicGuidebook.json (39 rules loaded)
   [10/80] abstracts evaluated ( 12.5%)...
   [20/80] abstracts evaluated ( 25.0%)...
   [30/80] abstracts evaluated ( 37.5%)...
   [40/80] abstracts evaluated ( 50.0%)...
   [50/80] abstracts evaluated ( 62.5%)...
   [60/80] abstracts evaluated ( 75.0%)...
   [70/80] abstracts evaluated ( 87.5%)...
   [80/80] abstracts evaluated (100.0%)...

✅ Jev Semantic Classification complete in 53.78s (0.67s/doc, 36.0ms/span)!

📊 Jev Independent Prediction Distribution:
  • O                                               :  429 ( 28.8%)
  • Nutrition                                       :  366 ( 24.5%)
  • Sleep                                           :  152 ( 10.2%)
  • Environmental_exposures                         :  135 (  9.0%)
  • Physical_activities                             :  133 (  8.9%)
  • Substance_use                                   :  109 (  7.3%)

In [4]:
# ==============================================================================
# [CELL 4]: Dual-Model Gating & Workload Analysis (Gating View)
# ==============================================================================
# Compares SpanNER proposal labels with independent Jev semantic classifications.
# Partitions spans into:
#   1. Consensus Fast-Path: Model agreement -> Auto-accepted with 0 LLM cost.
#   2. Escalated Subset   : Discrepancies, high uncertainty, or novelty -> Sent to arbitration.
GATING_MODE = 'discrepancy'       # 'discrepancy' (default), 'referee', or 'composite'
MIN_CONSENSUS_CONF = 0.35
MAX_UNCERTAINTY = 0.75
NOVELTY_THRESHOLD = 0.55
ENABLE_MARGIN_VETO = True

gating = JeBertGating(
    gating_mode=GATING_MODE,
    min_consensus_conf=MIN_CONSENSUS_CONF,
    max_uncertainty_threshold=MAX_UNCERTAINTY,
    novelty_threshold=NOVELTY_THRESHOLD,
    enable_margin_safeguard_veto=ENABLE_MARGIN_VETO,
)

consensus_cands_by_doc = {}
escalated_cands_by_doc = {}
discrepancy_pairs = Counter()
tot_consensus = tot_escalated = 0

for doc_id in test_candidates_cache:
    s_cands = spanner_preds_by_doc[doc_id]
    j_preds = jev_preds_by_doc[doc_id]
    
    esc, cons = gating.filter_candidates(s_cands, j_preds)
    consensus_cands_by_doc[doc_id] = cons
    escalated_cands_by_doc[doc_id] = esc
    tot_consensus += len(cons)
    tot_escalated += len(esc)
    
    for c in esc:
        if c.get('is_discrepancy', False):
            pair = (c.get('spanner_label', 'O'), c.get('jev_label', 'O'))
            discrepancy_pairs[pair] += 1

tot_spans = tot_consensus + tot_escalated
print('=' * 80)
print('📊 JeBert Dual-Model Gating & Workload Summary:')
print(f'   • Total Candidate Spans Evaluated : {tot_spans}')
print(f'   • Consensus Auto-Accepted (0 Cost): {tot_consensus:4d} ({tot_consensus / max(1, tot_spans) * 100:5.1f}%)')
print(f'   • Escalated for Review/Arbitration: {tot_escalated:4d} ({tot_escalated / max(1, tot_spans) * 100:5.1f}%)')
print(f'   • Model Label Discrepancies       : {sum(discrepancy_pairs.values()):4d} ({sum(discrepancy_pairs.values()) / max(1, tot_spans) * 100:5.1f}%)')
print('=' * 80)

print('\n🔍 Top Model Label Discrepancies (SpanNER vs Jev):')
print(f'   {"SpanNER Hypothesis":<32} | {"Jev Hypothesis":<32} | {"Count":<6}')
print('   ' + '-' * 76)
for (s_l, j_l), cnt in discrepancy_pairs.most_common(8):
    print(f'   {s_l:<32} | {j_l:<32} | {cnt:5d}')


📊 JeBert Dual-Model Gating & Workload Summary:
   • Total Candidate Spans Evaluated : 1492
   • Consensus Auto-Accepted (0 Cost):  710 ( 47.6%)
   • Escalated for Review/Arbitration:  782 ( 52.4%)
   • Model Label Discrepancies       :  477 ( 32.0%)

🔍 Top Model Label Discrepancies (SpanNER vs Jev):
   SpanNER Hypothesis               | Jev Hypothesis                   | Count 
   ----------------------------------------------------------------------------
   Nutrition                        | O                                |   129
   Substance_use                    | O                                |    70
   Socioeconomic_factors            | O                                |    65
   Sleep                            | O                                |    44
   Mental_health_practices          | O                                |    42
   Environmental_exposures          | O                                |    27
   Personal_care_products_and_cosmetic_procedures | O            

In [5]:
# ==============================================================================
# [CELL 5]: Escalation Arbitration & Margin Safeguard Veto (Arbitration View)
# ==============================================================================
# Arbitration is ON by default.
# Performs a fast 0.5s connection health check to the LLM backend:
#   - If LLM server is active: invokes full heavy LLM arbitration.
#   - If LLM server is offline: smoothly engages the Fast Dual-Model Referee (0ms latency, zero hang).
# Margin Safeguard Veto protects high-confidence SpanNER spans from drops to Non-LSF (O).
RUN_ARBITRATION = True              # Default: ON
ARBITRATOR_MODE = 'ace_generator'   # 'ace_generator', 'discovery', or 'dinasor_dingen'
LLM_BACKEND = 'api'                 # 'api', 'openai', 'lmstudio', 'ollama', 'dummy'
LLM_MODEL = 'gemini-cli'            # 'gemini-cli', 'gpt-4o-mini', etc.
LLM_BASE_URL = os.getenv('LLM_API_BASE_URL', os.getenv('OPENAI_BASE_URL', 'http://localhost:20128/v1'))

def _probe_llm_server(base_url: str, timeout: float = 1.0) -> bool:
    try:
        r = httpx.get(f'{base_url.rstrip("/")}/models', timeout=timeout)
        return r.status_code < 500
    except Exception:
        return False

llm_available = _probe_llm_server(LLM_BASE_URL) if (RUN_ARBITRATION and LLM_BACKEND in ['api', 'lmstudio', 'ollama']) else RUN_ARBITRATION
if llm_available:
    print(f'🚀 Active Heavy LLM Arbitrator connected at: {LLM_BASE_URL} (Model: {LLM_MODEL})')
else:
    print(f'⚡ LLM server at {LLM_BASE_URL} is offline -> Engaging Fast Dual-Model Referee (0ms latency, zero hang).')

jebert_pipeline = JeBertPipeline(
    jev_mode=JEV_MODE,
    gating_mode=GATING_MODE,
    arbitrator_mode=ARBITRATOR_MODE,
    enable_llm_arbitration=llm_available,
    enable_margin_safeguard_veto=ENABLE_MARGIN_VETO,
)

t0_arb = time.time()
jebert_final_preds_by_doc = {}
veto_count = 0

for doc_id, doc_info in test_candidates_cache.items():
    abstract_text = doc_info.get('abstract', doc_info.get('text', ''))
    cons_cands = consensus_cands_by_doc[doc_id]
    esc_cands = escalated_cands_by_doc[doc_id]
    
    # 1. Accept consensus candidates directly (0 cost)
    final_list = []
    for c in cons_cands:
        lbl = canonicalize_label(c.get('jev_label', c.get('spanner_label', 'O')), use_new_labels=True)
        c_dict = dict(c)
        c_dict['label'] = lbl
        c_dict['final_label'] = lbl
        c_dict['source'] = 'JeBert-Consensus'
        final_list.append(c_dict)
        
    # 2. Arbitrate escalated candidates
    if esc_cands:
        arb_resolved = jebert_pipeline.arbitrate_escalated(
            abstract=abstract_text,
            escalated_cands=esc_cands,
            use_llm=llm_available,
            use_fast_referee_fallback=True,
        )
        for s in arb_resolved:
            if 'VETO' in s.get('source', ''):
                veto_count += 1
            final_list.append(s)
            
    final_list.sort(key=lambda x: (x.get('start_char', 0), x.get('end_char', 0)))
    jebert_final_preds_by_doc[doc_id] = final_list

elapsed_arb = time.time() - t0_arb
total_final = sum(len(v) for v in jebert_final_preds_by_doc.values())
print(f'\n✅ Arbitration & Assembly complete in {elapsed_arb:.2f}s across {len(jebert_final_preds_by_doc)} abstracts!')
print(f'   • Final Candidate Spans Preserved: {total_final}')
print(f'   • Margin Safeguard Vetoes Applied : {veto_count} (Restored high-margin SpanNER spans over drops to O)')


⚡ LLM server at http://localhost:20128/v1 is offline -> Engaging Fast Dual-Model Referee (0ms latency, zero hang).

✅ Arbitration & Assembly complete in 0.00s across 80 abstracts!
   • Final Candidate Spans Preserved: 1492
   • Margin Safeguard Vetoes Applied : 0 (Restored high-margin SpanNER spans over drops to O)


In [6]:
# ==============================================================================
# [CELL 6]: Standardized Dual-View Evaluation Benchmark
# ==============================================================================
# Evaluates JeBert across both standardized views:
#   View 1: Candidate Arbitration Benchmark (Decision accuracy on proposed mentions)
#   View 2: Global End-to-End Dataset Benchmark (Strict CoNLL, Partial MUC-7, Mention-level)

eval_preds = []
eval_gold = []

for doc_id, doc_info in test_candidates_cache.items():
    preds = jebert_final_preds_by_doc.get(doc_id, [])
    # Filter out Non-LSF (O) from final positive entity predictions for dataset evaluation
    ents = [p for p in preds if p.get('final_label', p.get('label', 'O')) != 'O']
    eval_preds.append(ents)
    eval_gold.append(doc_info.get('ground_truth', []))

print('=' * 96)
print('🏆 DUAL-VIEW EVALUATION BENCHMARK FOR JEBERT')
print('=' * 96)

benchmark_results = evaluate_ner_dual_view(
    predicted_entities=eval_preds,
    ground_truth_entities=eval_gold,
    candidate_proposals=[v.get('candidates', []) for v in test_candidates_cache.values()],
    system_name='JeBert (SpanNER + Jev + Gating + Referee)',
)


🏆 DUAL-VIEW EVALUATION BENCHMARK FOR JEBERT
🏆 VIEW 1: CANDIDATE-LEVEL ARBITRATION BENCHMARK (Decision Accuracy on Proposed Mentions)
   Evaluates decision accuracy on candidate mentions (SpanNER Proposer -> JeBert (SpanNER + Jev + Gating + Referee))
Class Name                                       | SpanNER F1 | JeBert (SpanNER + Jev + Gating + Referee) F1 | Delta    | TP    | FP    | FN   
------------------------------------------------------------------------------------------------------------------
Environmental_exposures                          |    61.22% |      63.11% | +1.89%   | 77    | 86    | 4    
Mental_health_practices                          |    75.29% |      75.29% | +0.00%   | 32    | 10    | 11   
Non-LSF                                          |     0.00% |       0.00% | +0.00%   | 0     | 0     | 77   
Non_physical_leisure_time_activities             |    62.39% |      64.29% | +1.90%   | 36    | 38    | 2    
Nutrition                                        | 

In [7]:
# ==============================================================================
# [CELL 7]: Model Discrepancy & Margin Safeguard Veto Deep Dive Inspector
# ==============================================================================
# Inspects real cases where SpanNER and Jev had conflicting classifications,
# and reveals how the dual-model gating and arbitrator resolved the entity.
discrepancies = []
vetoed_cases = []

for doc_id, preds in jebert_final_preds_by_doc.items():
    for p in preds:
        if p.get('is_discrepancy', False):
            discrepancies.append((doc_id, p))
        if 'VETO' in p.get('source', ''):
            vetoed_cases.append((doc_id, p))

print(f'🔍 Total Model Discrepancies Analyzed: {len(discrepancies)}')
print(f'🛡️ Total Margin Safeguard Vetoes     : {len(vetoed_cases)}\n')

print('=' * 88)
print('Case Inspection: Top Model Discrepancy Resolutions:')
print('=' * 88)
for idx, (doc_id, p) in enumerate(discrepancies[:8], 1):
    s_text = p.get('span_text', '')
    s_lbl = p.get('spanner_label', 'O')
    j_lbl = p.get('jev_label', 'O')
    f_lbl = p.get('final_label', 'O')
    src = p.get('source', '')
    u_score = p.get('uncertainty', 0.0)
    margin = p.get('margin', 0.0)
    j_conf = p.get('jev_confidence', 0.0)
    
    print(f'Case {idx} [{doc_id}]: "{s_text}"')
    print(f'   • SpanNER Hypothesis : {s_lbl:<32} (margin={margin:.2f}, uncertainty={u_score:.2f})')
    print(f'   • Jev Hypothesis     : {j_lbl:<32} (confidence={j_conf:.2f})')
    print(f'   • Resolved Outcome   : {f_lbl:<32} (Source: {src})')
    print('-' * 88)


🔍 Total Model Discrepancies Analyzed: 477
🛡️ Total Margin Safeguard Vetoes     : 0

Case Inspection: Top Model Discrepancy Resolutions:
Case 1 [10398936]: "job history"
   • SpanNER Hypothesis : Socioeconomic_factors            (margin=0.82, uncertainty=0.14)
   • Jev Hypothesis     : O                                (confidence=0.89)
   • Resolved Outcome   : Socioeconomic_factors            (Source: JeBert-SpanNER-Margin-Referee)
----------------------------------------------------------------------------------------
Case 2 [10398936]: "job history. Physical activity"
   • SpanNER Hypothesis : Socioeconomic_factors            (margin=0.31, uncertainty=0.48)
   • Jev Hypothesis     : O                                (confidence=0.53)
   • Resolved Outcome   : O                                (Source: JeBert-DualModel-Background)
----------------------------------------------------------------------------------------
Case 3 [10398936]: "jobs"
   • SpanNER Hypothesis : Socioeconomic_fac

In [ ]:
# ==============================================================================
# [CELL 8]: Export Structured JeBert Run Reports, JSON Metrics & Abstract Audit Logs
# ==============================================================================
# Mirrors the BERTized ACE runs directory structure under output/jebert/runs/:
#   output/jebert/runs/run_<timestamp>_jebert/
#     ├── dynamicGuidebook.json
#     ├── run_summary_metrics.json
#     ├── run_summary_report.txt
#     └── abstracts/
#         └── test/
#             └── <doc_id>.txt
# ==============================================================================
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder_name = f"run_{timestamp_str}_jebert"
base_export_dir = JEBERT_RUNS_DIR / run_folder_name
abstracts_export_dir = base_export_dir / "abstracts"
test_abstracts_dir = abstracts_export_dir / "test"

test_abstracts_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 0. Snapshot Dynamic Guidebook into Run Folder
# ------------------------------------------------------------------------------
guidebook_snapshot_dest = base_export_dir / "dynamicGuidebook.json"
if os.path.exists(GUIDEBOOK_PATH):
    shutil.copy2(GUIDEBOOK_PATH, guidebook_snapshot_dest)
    print(f"  • Dynamic Guidebook: Snapshot saved to {guidebook_snapshot_dest.name}")
elif (PROJECT_ROOT / "output" / "bertized_ace" / "dynamicGuidelines.json").exists():
    shutil.copy2(PROJECT_ROOT / "output" / "bertized_ace" / "dynamicGuidelines.json", guidebook_snapshot_dest)
    print(f"  • Dynamic Guidebook: Fallback snapshot saved to {guidebook_snapshot_dest.name}")

print("=" * 96)
print("💾 EXPORTING JEBERT STRUCTURED RUN REPORTS & ABSTRACT AUDIT LOGS")
print("=" * 96)
print(f"  • Output Directory : {base_export_dir}")
print(f"  • Total Abstracts  : {len(test_candidates_cache)}")

# ------------------------------------------------------------------------------
# 1. Save Per-Abstract TXT Audit Files
# ------------------------------------------------------------------------------
all_preds_flat = []
all_gts_flat = []

for doc_id, doc_info in test_candidates_cache.items():
    abstract_text = doc_info.get("abstract", doc_info.get("text", ""))
    gt_ents = doc_info.get("ground_truth", [])
    preds = jebert_final_preds_by_doc.get(doc_id, [])
    
    log_jebert_abstract_audit(
        doc_id=doc_id,
        text=abstract_text,
        gt_ents=gt_ents,
        preds=preds,
        abstracts_dir=str(test_abstracts_dir),
        timestamp_str=timestamp_str,
        architecture_str=f"JeBert ({JEV_MODE} + {GATING_MODE} + {ARBITRATOR_MODE})",
        arbitrator_mode=ARBITRATOR_MODE,
        use_new_labels=True,
    )
    all_preds_flat.append(preds)
    all_gts_flat.append(gt_ents)

print(f"  • Abstract Audits  : Generated {len(test_candidates_cache)} audit reports in abstracts/test/")

# ------------------------------------------------------------------------------
# 2. Canonical 10-Outcome Diagnostic Breakdown
# ------------------------------------------------------------------------------
audit_breakdown = compute_canonical_audit_breakdown(all_preds_flat, all_gts_flat, use_new_labels=True)
audit_table_str = format_canonical_audit_table(audit_breakdown, method="JeBert (Gated)", use_new_labels=True)

# ------------------------------------------------------------------------------
# 3. Save run_summary_metrics.json
# ------------------------------------------------------------------------------
tot_discrepancies = sum(discrepancy_pairs.values()) if "discrepancy_pairs" in globals() else 0
cand_bench = benchmark_results.get("view1_candidate_arbitration", {})
e2e_bench = benchmark_results.get("view2_end_to_end", {})

metrics_json = {
    "architecture": "JeBert_Pipeline",
    "jev_mode": JEV_MODE,
    "gating_mode": GATING_MODE,
    "arbitrator_mode": ARBITRATOR_MODE,
    "enable_margin_safeguard_veto": ENABLE_MARGIN_VETO,
    "timestamp": timestamp_str,
    "total_test_abstracts": len(test_candidates_cache),
    "gating_statistics": {
        "total_candidate_spans": tot_spans,
        "consensus_fast_path": tot_consensus,
        "escalated_spans": tot_escalated,
        "escalation_rate_pct": (tot_escalated / max(1, tot_spans)) * 100.0,
        "discrepancy_count": tot_discrepancies,
        "margin_vetoes_applied": veto_count,
    },
    "dual_view_benchmark": benchmark_results,
    "canonical_10_outcomes": audit_breakdown,
}

with open(base_export_dir / "run_summary_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_json, f, indent=4)

# ------------------------------------------------------------------------------
# 4. Save & Display run_summary_report.txt
# ------------------------------------------------------------------------------
report_lines = [
    "=" * 105,
    "📊 JEBERT DUAL-MODEL EXPERIMENT RUN SUMMARY REPORT",
    f"Timestamp: {timestamp_str} | Total Test Abstracts: {len(test_candidates_cache)} | Mode: {JEV_MODE.upper()}",
    f"Gating: {GATING_MODE} | Arbitrator: {ARBITRATOR_MODE} | Margin Veto: {ENABLE_MARGIN_VETO}",
    "=" * 105,
    "",
    "1. GATING WORKLOAD & ARBITRATION DYNAMICS",
    "-" * 105,
    f"  • Total Candidate Spans       : {tot_spans}",
    f"  • Consensus Fast-Path (0 Cost): {tot_consensus} ({tot_consensus / max(1, tot_spans) * 100:.1f}%)",
    f"  • Escalated to Arbitrator     : {tot_escalated} ({tot_escalated / max(1, tot_spans) * 100:.1f}%)",
    f"  • Label Discrepancies         : {tot_discrepancies}",
    f"  • Margin Safeguard Vetoes     : {veto_count} (Protected high-margin SpanNER predictions)",
    "",
    "2. CANDIDATE ARBITRATION BENCHMARK (All Proposed Mentions)",
    "-" * 105,
    f"  • SpanNER Baseline : Micro F1 = {cand_bench.get('spanner_micro_f1', 0.0)*100:.2f}% | Macro F1 = {cand_bench.get('spanner_macro_f1', 0.0)*100:.2f}%",
    f"  • JeBert Dual-Model: Micro F1 = {cand_bench.get('micro_f1', 0.0)*100:.2f}% | Macro F1 = {cand_bench.get('macro_f1', 0.0)*100:.2f}%",
    f"  • Net Delta        : Micro F1 = {(cand_bench.get('micro_f1', 0.0) - cand_bench.get('spanner_micro_f1', 0.0))*100:+.2f}% | Macro F1 = {(cand_bench.get('macro_f1', 0.0) - cand_bench.get('spanner_macro_f1', 0.0))*100:+.2f}%",
    "",
    "3. GLOBAL END-TO-END DATASET BENCHMARK (All Gold Entities)",
    "-" * 105,
    f"{'Formulation':<25} | {'Precision':<12} | {'Recall':<12} | {'F1-Score':<12} | {'TP / FP / FN':<15}",
    "-" * 105,
]

for form_key, form_name in [
    ("strict_conll", "Strict CoNLL (1)"),
    ("ace_nested", "ACE Nested (2)"),
    ("partial_muc7", "Partial MUC-7 (3)"),
    ("candidate_mention", "Candidate Mention (4)")
]:
    f_res = e2e_bench.get(form_key, {})
    report_lines.append(
        f"{form_name:<25} | {f_res.get('p', 0)*100:>10.2f}% | {f_res.get('r', 0)*100:>10.2f}% | {f_res.get('f1', 0)*100:>10.2f}% | {f_res.get('tp', 0):>4}/{f_res.get('fp', 0):>4}/{f_res.get('fn', 0):>4}"
    )

report_lines.extend([
    "",
    "-" * 105,
    "4. CANONICAL 10-OUTCOME DIAGNOSTIC BREAKDOWN",
    "-" * 105,
    audit_table_str,
    "=" * 105
])

with open(base_export_dir / "run_summary_report.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print(f"\n✅ Successfully exported JeBert run reports, metrics, and abstract logs to:\n   {base_export_dir}")
print("\n" + "\n".join(report_lines))

